<div style="border-left:8px solid #BC8F8F;background:#ffffff;padding:14px 18px;border-radius:4px"><div style="font:600 13px sans-serif;color:#BC8F8F;letter-spacing:.08em">PART A</div><div style="font:600 28px sans-serif;margin-top:2px;color:#1f2430">Presentation</div><div style="font:15px sans-serif;margin-top:6px;color:#1f2430">Watch and follow along. Searching a collection by what images look like.</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 1 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

# Notebook 3: Searching by what images look like, and looking closer

The Europeana section of the intro notebook found objects through their *metadata*. This notebook searches by **content**: you describe an image in words, or give an image, and get the most similar ones from a shared collection.

It rests on an **embedding**: a list of numbers (here 512) that a neural network computes from an image, so that images with similar content get similar numbers. [CLIP](https://openai.com/research/clip) computes embeddings for images *and* short texts in the same space, so a sentence can be compared directly with every image.

In [ ]:
# Setup: run this cell first.
import numpy as np
from IPython.display import display
from clip_helpers import *
from region_helpers import *
from iiif_helpers import show_mirador_many, url_panel

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 2 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How the dataset was made

The dataset contains **no images**, only numbers and links. It was prepared once by the instructor:

```
Europeana Search API  →  metadata records (title, creator, institution, rights, links)
        │   for every record: request a small version of the image
        │      from its IIIF image service, or from Europeana's thumbnail service
        ▼
CLIP image encoder  →  one vector of 512 numbers per image   (the image is then discarded)
```

Only openly reusable images from Europeana's **Art** collection are included. The **originals and the small versions stay at the institutions**: when a search returns results, your computer fetches *those* images from their URLs, and nothing else.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 3 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## The collection

`metadata.csv` and the vectors are in the same order: row *i* describes vector *i*. A random sample (the images are fetched from their institutions now):

In [ ]:
coll = Collection()
coll.summary()

rng = np.random.default_rng(1)
coll.show_grid(rng.choice(len(coll.meta), 12, replace=False))

The Art collection is broader than paintings: it also holds drawings, prints, photographs, decorative objects, coins, maps and ethnographic items. Who contributes the most?

In [ ]:
coll.top_institutions_and_countries()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 4 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## CLIP: text and images in one space

![CLIP contrastive pre-training: a text encoder and an image encoder produce vectors that are compared in a matrix](../misc/assets/img/clip.svg)

<small>Contrastive pre-training of CLIP: matching image-text pairs are pulled together (blue diagonal), all other pairs pushed apart. Figure: OpenAI, ["CLIP: Connecting text and images"](https://openai.com/index/clip/) (Radford et al., 2021).</small>

The image half was applied to the whole collection beforehand. For a query only the **text encoder** runs, which is fast on a laptop.

In [ ]:
clip_model = Clip()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 5 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with text

The query becomes a vector in the same 512-dimensional space. Both are normalised to length 1, so their **dot product** is their cosine similarity: 1 means the same direction, near 0 unrelated. Ranking the whole collection is one matrix–vector product.

In [ ]:
QUERY = "a ship on a stormy sea"

scores = coll.similarity(clip_model.text(QUERY))
best = coll.top_k(scores, 12)
coll.show_grid(best, scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 6 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Restricting the search with a mask

Because the metadata is aligned with the vectors, a filter is just a boolean **mask** applied before ranking. Here: only images dated before 1900.

In [ ]:
QUERY = "a portrait of a person"
mask = (coll.year > 0) & (coll.year < 1900)                          # dated, and before 1900
# mask = np.isin(coll.country, ["Sweden", "Finland", "Norway"])      # alternative: by country
# mask = np.isin(coll.institution, ["Wellcome Collection"])          # alternative: by institution

scores = coll.similarity(clip_model.text(QUERY))
print(f"{mask.sum():,} of {len(mask):,} images pass the filter. Top results for {QUERY!r}:\n")
coll.show_grid(coll.top_k(scores, 12, mask), scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 7 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Where do the results come from?

A result is only as varied as the collection it searches. Comparing the institutions in the top results with those in the whole collection shows whether a query favours some sources.

In [ ]:
QUERY = "a ship on a stormy sea"
scores = coll.similarity(clip_model.text(QUERY))
top = coll.top_k(scores, 100)
coll.institution_share(top, QUERY)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 8 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with an image

The same space holds image embeddings, so an image can be the query. Here: the best match of the ship query, used as "more like this".

In [ ]:
SEED = int(top[0])
print("Query: collection image", SEED, "-", coll.meta[SEED]["title"][:60])
display(coll.small_image(SEED))

scores = coll.similarity(coll.V[SEED].astype(np.float32))
scores[SEED] = -1                          # do not return the query itself
coll.show_grid(coll.top_k(scores, 12), scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 9 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Look closely: a larger version, on request

The small versions are enough for CLIP, not for studying a detail. Once a result is interesting, a larger version or a detail can be requested from the institution:

- **With a IIIF image service**: one URL returns a detail at the size you ask for, and nothing else is downloaded.
- **Without one**: the whole original file has to be downloaded, often several megabytes.

First a result with an image service: the central quarter of the image at 1,000 pixels.

In [ ]:
QUERY = "a ship on a stormy sea"
scores = coll.similarity(clip_model.text(QUERY))
iiif_hits = coll.top_k(scores, 6, coll.has_iiif)

coll.show_grid(iiif_hits, scores)
coll.look_closely(int(iiif_hits[0]), size=1000, region=(0.25, 0.25, 0.5, 0.5))

Then the best match overall, which may have no image service. Compare the amount of data transferred:

In [ ]:
overall = int(coll.top_k(scores, 1)[0])
coll.look_closely(overall, size=1000)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 10 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Looking closer: regions and details

So far we turned **whole images** into vectors. But an image is rarely about one thing: a banknote is a banknote, and it also shows a town, a soldier, a woman with a basket, a number. One vector per image can only average them.

This section cuts an image into **regions**, embeds each region on its own, and searches the shared collection with it. Then you compare: what does the collection answer to the whole image, and what to a detail?

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 11 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## A region is a URL

IIIF can deliver any rectangle of an image. The region is part of the URL, in pixels of the full image: `x,y,width,height`.

```
{image service}/{region}/{size}/{rotation}/{quality}.{format}
 .../iiif/abc123 / 850,894,1700,1788 / !448,448 / 0 / default.jpg
                    ^ the central quarter       ^ at most 448 px: the size CLIP reads
```

For the model, a region is simply **another image**. Only those pixels travel from the institution's server. The same rectangle is also how an annotation points at a detail: `canvas#xywh=850,894,1700,1788`. And the **quality** is part of the URL too: `default`, `gray` or `bitonal`.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 12 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## One picture, six regions

An emergency banknote (*Notgeld*, 20 Heller, Waizenkirchen) of the Salzburg Museum, with a IIIF image service. `frac(x, y, w, h)` gives a region as fractions of the picture (0 to 1), converted to pixels.

In [ ]:
ID = "/1387/https___id_kulturpool_at_013cf817_e57a_4289_96b3_2a068c7fe756_cho"      # a record of the collection (Europeana ID)
row = next(i for i, r in enumerate(coll.meta) if r["id"] == ID)
pic = Picture(row, coll)

regions = {
    "town":       pic.frac(0.10, 0.28, 0.80, 0.30),
    "soldier":    pic.frac(0.12, 0.52, 0.20, 0.42),
    "woman":      pic.frac(0.74, 0.55, 0.15, 0.38),
    "numeral 20": pic.frac(0.38, 0.56, 0.27, 0.36),
    "title":      pic.frac(0.22, 0.02, 0.60, 0.12),
    "corner":     pic.frac(0.00, 0.00, 0.18, 0.20),
}
pic.show(regions)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 13 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Whole image or region: what does the collection answer?

The same search with seven queries: the whole banknote and the six regions. Each row is one query, each card one of the 8 most similar images of the collection (the banknote itself is left out).

In [ ]:
exp = Experiment(coll, clip_model, pic)
exp.add("whole note")
for name, region in regions.items():
    exp.add(name, region)

exp.run(k=8)
exp.show()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 14 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Count what you expect

Looking is slow; a count is quick. Here we expect banknotes: how many of the 8 results of each query are banknotes (by title)? A **hypothesis you can count** is a good way to read a comparison.

In [ ]:
def is_note(i):
    t = coll.meta[int(i)]["title"]
    return "Notgeld" in t or "Heller" in t or "banknot" in t.lower()

for name in exp.queries:
    n = sum(is_note(i) for i in exp.top(name))
    print(f"{name:12s} {'#' * n}{'.' * (8 - n)}  {n} of 8 are banknotes")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 15 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How different are the answers?

The table counts how many results two queries share. The whole note shares most with itself and with the numeral (the part that says "money"). The town, the soldier and the woman answer with something else: landscapes, figures, saints. Is that a *better* answer, or just a different one? That is for you to judge by looking.

In [ ]:
exp.overlap()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 16 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Colour, grey, black and white

The IIIF **quality** is part of the URL, so the same region can be requested as `default`, `gray` or `bitonal` (pure black and white). Does the model still recognise it? A scan from a bitonal microfilm, or a photocopy, is such an image.

In [ ]:
q = Experiment(coll, clip_model, pic)
for quality in ("default", "gray", "bitonal"):
    q.add(f"whole, {quality}", quality=quality)
    q.add(f"soldier, {quality}", regions["soldier"], quality=quality)

q.run(k=8)
q.show()
q.overlap()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 17 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Regions against text

A text query lives in the same space, so it can be added as one more row: does the region find what the *words* find?

In [ ]:
exp.add("text: banknote", text="a banknote")
exp.add("text: soldier", text="a soldier with a cannon")
exp.run(k=8)
exp.show()
exp.overlap()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 18 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Cut everything: tiles

Instead of choosing regions by hand, cut the whole picture into overlapping **tiles**, embed every tile, and search the tiles with a text. The result is a map of where in the picture something is. With IIIF each tile is one small request, so even a very large picture is possible (you will find large pictures in Part B).

In [ ]:
scan = TileScan(pic, clip_model, rows=4, overlap=0.5)
found = scan.show("a soldier with a cannon", k=3)

In [ ]:
other = scan.show("a woman with a basket", k=3)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 19 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Annotations: pointing at regions

Every region becomes a **web annotation** that targets the canvas with `#xywh=`. The manifest also lists the images found for each region as canvases, grouped in one range per region. Open it in Mirador: the annotations are in the side panel, the images in the table of contents.

In [ ]:
notes = [
    note(regions["town"],       "The town. The collection answers with landscapes, not with banknotes.", hits=exp.top("town", 3)),
    note(regions["soldier"],    "The soldier with a cannon. Figures and prints: no banknote among the results.", hits=exp.top("soldier", 3)),
    note(regions["numeral 20"], "The numeral. The only detail that still finds banknotes: it says 'money'.", hits=exp.top("numeral 20", 3)),
]
notes += [note(r, f"a soldier with a cannon? (tile score {s:.2f})") for r, s in found[:2]]

manifest = build_manifest("Demo: a banknote and its details", pic, notes, coll, summary="Regions of one banknote and the images found for them.")
show_manifest(manifest)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 20 / 20</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## The presentation

A `Deck` collects slides from your own results: text, the picture with its regions, the comparison, found images, and the annotated viewer. It is one HTML file (arrow keys to move). This is what you will build in Part B.

In [ ]:
deck = Deck("Demo: a banknote and its details", author="Workshop", subtitle="Whole image vs. region")
deck.add_text("Question", "Does the collection answer a detail differently from the whole picture?")
deck.add_picture("Picture and regions", pic, regions)
deck.add_compare("Whole note vs. regions", exp, text="- The whole note finds banknotes.\n- The town, the soldier and the woman find something else.")
deck.add_mirador("Annotated", manifest)
deck.show()

<div style="border-left:8px solid #FF7F50;background:#ffffff;padding:14px 18px;border-radius:4px"><div style="font:600 13px sans-serif;color:#FF7F50;letter-spacing:.08em">PART B</div><div style="font:600 28px sans-serif;margin-top:2px;color:#1f2430">Hands-on</div><div style="font:15px sans-serif;margin-top:6px;color:#1f2430">Your turn: a short exercise, then your own presentation (5 minutes each).</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 1 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Exercise: build your own selection

Choose a **theme**, find matching images with CLIP, and look at your selection in one IIIF viewer. Then share it.

1. Pick a theme and write a **query** for it, such as `"a cat"`, `"a ship on a stormy sea"`, `"gold coins"`, `"a map of a city"` or `"a faded black and white photograph"`.
2. Optionally restrict it with a **mask** (period, country or institution).
3. Run the cell, then note the **numbers** of 3 to 5 results you like.

In [ ]:
# ============================================================
# EXERCISE: your theme
# ============================================================
QUERY = "a ship on a stormy sea"      # <-- your theme, described in words (English works best)
mask  = None                          # <-- or e.g. (coll.year > 0) & (coll.year < 1900)
                                      #     or np.isin(coll.country, ["Sweden", "Finland"])
# ============================================================

scores = coll.similarity(clip_model.text(QUERY))
best = coll.top_k(scores, 12, mask)
coll.show_grid(best, scores, numbered=True)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 2 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Exercise: from your picks to a viewer

Enter the numbers of the results you chose. The cell builds the **IIIF manifest URL** of each record (the pattern from the intro notebook), checks that it answers, and prints a block to paste into the shared pad.

In [ ]:
PICKS = [0, 1, 2]        # <-- the numbers of the results you chose, e.g. [0, 3, 5, 8]

chosen = [int(best[n]) for n in PICKS]
urls = [manifest_url(coll.meta[i]["id"]) for i in chosen]

print("Checking the manifests ...")
for i, u in zip(chosen, urls):
    ok = requests.get(u, timeout=30).ok
    print(("✓" if ok else "✗"), coll.meta[i]["title"][:60], "|", coll.meta[i]["institution"][:30])

print("\n----- paste into the shared pad -----")
print("Theme:", QUERY)
for u in urls:
    print(u)

In [ ]:
show_mirador_many(urls)

### Stretch

- Pick one result that has an image service and look at a **detail**: `coll.look_closely(chosen[0], region=(0.25, 0.25, 0.5, 0.5))`. Change the region (fractions `x, y, width, height`) to frame something you want to point out.
- Search with **your own image**: see the Appendix (section C).
- Narrow the theme with a **mask**, for example only images from one country, and compare the results with the unmasked search.

<details><summary><b>Hints</b> (click to open)</summary>

- The numbers (`#0`, `#1`, ...) are printed on each card. They refer to the list of 12 results of the cell above, not to the collection.
- A ✗ means the record has no manifest. Pick another result and run the cell again.
- Descriptions work better than single words: `"an oil painting of flowers in a vase"` instead of `"flowers"`.
- If the viewer stays empty, check that the manifest URLs above start with `https://iiif.europeana.eu/presentation/`.
</details>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 3 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

# Your task: regions and details

Choose **one picture** and find out what a **detail** of it means for a search of the collection. Then present what you found.

**You decide the question.** For example:
- a picture from your own research, or a IIIF image from your institution
- a picture where the subject is small, or where the *background* is the interesting part
- a region that you think a human sees clearly, but the model may not (a hand, a script, a signature, a symbol)

**Deliverable** (handed in on the workshop's [Switch drive](https://drive.switch.ch/index.php/s/JeirWKGtP2HBxEm), see step 5):
- a **deck** of 4 to 6 slides (Part B, step 5)
- a IIIF **manifest** with at least 3 annotated regions
- optional: a screenshot, a few images, or any file you would like us to open (an `.html` page, for example)

**The 5 minutes:** the picture and your question (1 min), the whole image against your regions (2 min), **one limitation you found** (1 min), the annotated viewer (1 min). Say a few words about what you found, and how it might (or might not) relate to your own projects.

You can work alone or in a pair, whatever fits your project.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 4 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Step 1: choose a picture

Three ways to get a `Picture`:

- **from the collection**: a row number (use a text search, then pick a result), or search by title below
- **from IIIF**: the URL of any IIIF image service (the part before `/full/...`, or its `info.json`). Your institution's, Wellcome, the Internet Archive, ...
- **a file**: the path of an image next to the notebook

Pictures with an image service are best: a region then costs one small request instead of a full download. Pictures that are **large** have the most detail: `find_large(coll)` asks the image services of a random sample and shows the largest ones.

In [ ]:
# ============================================================
# STEP 1: your picture
# ============================================================
# a) from the collection: search by text, look at the numbers, pick one
QUERY = "a landscape with a town and a river"       # <-- what you are looking for
scores = coll.similarity(clip_model.text(QUERY))
best = coll.top_k(scores, 12, coll.has_iiif)         # only pictures with an image service
coll.show_grid(best, scores, numbered=True)

In [ ]:
# b) or look for large pictures (they have detail): the numbers are used in the next cell
large = find_large(coll, n=12, sample=300)       # <-- a different sample: seed=1, 2, ...   or mask=...

In [ ]:
SOURCE = int(best[0])        # <-- #0 of the results above is best[0], or large[0] for a large picture. Or: "https://.../iiif/xyz/info.json"  or  "my_picture.jpg"

pic = Picture(SOURCE, coll)
pic.show()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 5 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Step 2: define regions

A region is `(x, y, width, height)` in pixels, or use `pic.frac(x, y, w, h)` with fractions of the picture. Define 2 to 4 regions with names, run the cell, and adjust until the red boxes frame what you mean.

Tip: the `IIIF URL` printed below each region can be pasted into a browser tab.

In [ ]:
regions = {
    "left":   pic.frac(0.00, 0.20, 0.35, 0.60),     # <-- name: pic.frac(x, y, w, h)
    "centre": pic.frac(0.30, 0.25, 0.40, 0.50),
    "detail": pic.frac(0.60, 0.10, 0.15, 0.15),
}
pic.show(regions)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 6 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Step 3: whole image against regions

Run the same search with the whole picture and with each region. Look at the differences, and at the overlap table. Add a **text** query when you have a hypothesis ("is that a dog?").

Things to try:
- `mask=`: restrict the search to a country, an institution or a period (as in the search above)
- `quality="gray"` or `"bitonal"` on `exp.add(...)`: does the answer survive without colour?
- `fit="crop"`: the default of CLIP cuts a square from the middle of a wide region; `"pad"` keeps all of it
- a region that is much smaller or larger: at what size does the answer stop making sense?

In [ ]:
MASK = None        # <-- e.g. coll.year < 1900   or   np.isin(coll.country, ["Sweden", "Finland"])

exp = Experiment(coll, clip_model, pic, mask=MASK)
exp.add("whole image")
for name, region in regions.items():
    exp.add(name, region)
# exp.add("text", text="a dog")                     # <-- optional: a hypothesis in words
# exp.add("detail (cropped)", regions["detail"], fit="crop")

exp.run(k=10)
exp.show()
exp.overlap()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 7 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Step 4: annotate

Write one note per region: what is in it, and what the collection answered. Add `hits=` with the results you want to keep (their row numbers: `exp.top("name", 3)` gives the 3 best of a query). Then look at the result in Mirador.

Your notes are your **findings**. Say what you see, including where the model is wrong.

In [ ]:
notes = [
    note(regions["left"],   "What is here, and what did the collection answer?", hits=exp.top("left", 3)),
    note(regions["centre"], "...",                                              hits=exp.top("centre", 3)),
    note(regions["detail"], "...",                                              hits=exp.top("detail", 3)),
]

TITLE = "My title"       # <-- the title of your work
manifest = build_manifest(TITLE, pic, notes, coll, summary="One sentence: your question.")
show_manifest(manifest)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 8 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Step 5: build the deck and save it

Add the slides in the order you will present them. Every `add_` call is one slide. The deck opens below: click into it and use the arrow keys.

When it works, `save_submission` writes `index.html` and `manifest.json` into **`submissions/<NAME>/`** (in the file browser on the left, next to the notebooks). The shared folder is read-only, so you hand in through the Switch drive:

1. In the file browser, right-click each file and **Download** it.
2. Open the [Switch drive folder](https://drive.switch.ch/index.php/s/JeirWKGtP2HBxEm), create a **folder with your name** and drop your files in.
3. Add anything else you want to show: a screenshot, a few images, an `.html` page, whatever you like.

On the day, the files are opened from there.

In [ ]:
NAME   = "your-name"             # <-- your folder name in submissions/
AUTHOR = "Your Name"

deck = Deck(TITLE, author=AUTHOR, subtitle="Regions and details in Europeana")
deck.add_text("Question", "What did I want to find out?\n- one sentence\n- why this picture")
deck.add_picture("The picture and its regions", pic, regions, text="What I see in each region.")
deck.add_compare("Whole image vs. regions", exp, text="What changes, and what does not.")
deck.add_results("Best matches of one region", coll, exp.top("centre", 8), exp.results["centre"]["scores"], text="Which of these would a human accept?")
deck.add_text("A limitation", "- where the method failed\n- why I think so\n- what I would try next")
deck.add_mirador("Annotated", manifest)

deck.show()
save_submission(NAME, deck, manifest)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 9 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

### Stretch

- **Tiles on your own picture**: `scan = TileScan(pic, clip_model, rows=4, overlap=0.5)` then `scan.show("a dog", k=5)`. Turn the best tiles into notes with `note(region, text)`.
- **Mix region and text**: `v = exp.queries["detail"]["vec"] + 0.5 * clip_model.text("gold")`, then `exp.add_vector("detail + gold", v / np.linalg.norm(v))`.
- **Your own collection**: your IIIF images are `Picture(url)` too. Compare their regions with the Europeana sample.
- **Another view of the findings**: for one region, `coll.institution_share(exp.top("detail", 100), "detail")` shows which institutions answer. Is it the subject, or the way things were digitised?

<details><summary><b>Hints</b> (click to open)</summary>

- `original file not available`: many source links are dead. The notebook then uses the small version (about 400 px) and cuts regions from that: fine for searching, coarse for details.
- A `501` or an empty result from an image server: some servers deliver only certain sizes. The notebook retries with smaller sizes by itself.
- Regions smaller than about 5% of the picture side usually carry too little for CLIP. That is a finding, not a bug.
- Tile scans on a map or a page of text may prefer empty tiles. That is also a finding.
- `exp.run()` again after `exp.add(...)`, otherwise the new row has no results.
- Using a IIIF image from your institution: if `Picture(url)` fails, open `url/info.json` in a browser. If it does not show JSON, it is not the image service base URL.
</details>

<div style="margin-top:40vh;border-left:8px solid #C0C0C0;background:#ffffff;padding:14px 18px;border-radius:4px"><div style="font:600 13px sans-serif;color:#C0C0C0;letter-spacing:.08em">APPENDIX</div><div style="font:600 28px sans-serif;margin-top:2px;color:#1f2430">Background</div><div style="font:15px sans-serif;margin-top:6px;color:#1f2430">Details for reading afterwards. Not part of the presentation.</div></div>

## A. The dataset folder

| Path | Content |
|---|---|
| `metadata.csv` | one row per image: title, creator, year, institution, country, rights, the URL of the small version (`small_image_url`), the original file (`full_image_url`), and the IIIF image service when one answered |
| `embeddings/clip_vit_b32.npz` | the CLIP vectors, in the same order as `metadata.csv` |
| `models/` | the CLIP model weights |

For every record the small version was requested from the IIIF image service (448 px) when the media URL is one, and otherwise from Europeana's thumbnail service (400 px wide). A small version is all CLIP needs: it resizes every image to 224 × 224 pixels anyway. The helper functions are in `clip_helpers.py`.

## B. Writing queries

| Instead of | Try |
|---|---|
| `water` | `a river flowing through a landscape` |
| `old` | `a faded black and white photograph` |
| `art` | `an oil painting of flowers in a vase` |

CLIP understands descriptions better than single keywords, and it works best in English. It also matches *style and medium* ("a pencil drawing", "a photograph") as well as content, which is useful for a mixed collection.

## C. Search with your own image

Put an image file next to the notebook and give its name. Its embedding is computed with the image half of CLIP, on your computer.

In [ ]:
YOUR_IMAGE = None            # <-- e.g. "my_photo.jpg" (a file next to the notebook)

if YOUR_IMAGE:
    from PIL import Image
    query_image = Image.open(YOUR_IMAGE)
    display(query_image.copy().resize((200, int(200 * query_image.height / query_image.width))))
    scores = coll.similarity(clip_model.image(query_image))
    coll.show_grid(coll.top_k(scores, 12), scores)
else:
    print("Set YOUR_IMAGE to the name of an image file and run again.")

## D. What the region helpers do

| Call | Does |
|---|---|
| `find_large(coll)` | the largest pictures of a random sample of the collection |
| `Picture(source)` | opens an image: a collection row, a IIIF image service, a URL or a file; `.crop(region, quality=)`, `.region_url(region)`, `.frac(...)`, `.show(regions)` |
| `Experiment(...)` | `.add(name, region / text= / quality=)`, `.run(k)`, `.show()`, `.overlap()`, `.top(name, n)` |
| `TileScan(pic, clip_model, rows, overlap, quality)` | embeds overlapping tiles (`rows` tiles across the short side); `.best(query)`, `.show(query)` |
| `note(region, text, hits)` | one annotation |
| `build_manifest(title, pic, notes, coll)` | IIIF Presentation 3 manifest; `show_manifest(m)` |
| `Deck(title)` | `add_text`, `add_picture`, `add_compare`, `add_results`, `add_mirador`; `.show()`, `.save(path)` |
| `save_submission(name, deck, manifest)` | writes both into `submissions/<name>/` (then hand them in on the Switch drive) |

## E. Why `pad`?

CLIP reads a 224 x 224 square. Its own preprocessing resizes the short side and **cuts the centre square**: a wide region (a panorama strip) would lose both ends before the model sees it. `fit="pad"` (the default here) puts the whole region on a grey square instead. Neither is "right": compare them on a wide region.

## F. Quality

`gray` and `bitonal` are standard IIIF image qualities: the server converts. Servers that do not offer them answer with an error; the notebook then converts the region itself. `bitonal` is a hard threshold at mid-grey: on a line engraving or a printed text it keeps almost everything, on a photograph or a painting it destroys most of it.

## G. What the collection can and cannot answer

- The vectors were computed from **small versions of whole images** (about 400 px). A search with a region of an image *you* cut at 448 px compares a detail at high resolution with whole images at low resolution.
- A hit is never "the same object": it is the nearest vector. The images found for a region are neighbours in CLIP's space, which is shaped by its training data.
- Search results depend on what the collection contains: a sample of Europeana's open art images, not Europeana.

## H. The manifest

The manifest is IIIF Presentation 3: canvas 1 is your picture, with an `annotations` page of `commenting` annotations whose `target` is `canvas#xywh=x,y,w,h`. The images you found are further canvases (their small versions, with a link to the object page), and one range per annotated region lists them. Open `manifest.json` in any IIIF viewer, or host it and paste its URL into Mirador.

## I. Summary

- An **embedding** turns an image or a text into a vector. CLIP puts both in one space, so a sentence can be compared with images and an image with images.
- Searching a precomputed collection is a **matrix–vector product**, plus optional masks built from the metadata.
- The collection is a sample that reflects what institutions made available. Results show *what is in the collection*, so it pays to check where they come from.
- A **region** of an image is just another image: IIIF delivers any rectangle by URL. Searching with details shows that one vector per image averages what a picture shows.
- Small images are enough to compute embeddings. A larger version or a detail is requested **only when needed**, through the IIIF image API where the institution offers one, and as a full download where not.

### Limits to keep in mind

- CLIP was trained on images and captions from the web. It reflects the language, subjects and biases of that data, and works best on English queries and on photographic or pictorial content.
- Similarity is not meaning. Two images can be close because of composition, colour or medium and not because of subject.
- The dataset is a selection (openly reusable images from Europeana's Art collection), not Europeana as a whole. The Art collection is Europeana's own curation and is broader than fine art.
- The images live at the institutions. If a source is offline or has moved the file, that image shows as a grey square, and the links in `metadata.csv` reflect the time the dataset was built.
- Detecting IIIF services from the media URL finds many but not all of them. Some institutions have a service that cannot be seen from the URL, and are served through Europeana's thumbnails.